In [1]:
from pathlib import Path
_template = str(Path(__vsc_ipynb_file__).parent.parent / '../template.ipynb')
%run "$_template"

In [2]:
import lightgbm as lgb

In [3]:
df_test = pd.read_csv("../../data/modeling/test_encoded.csv")

oxidation = ['type_dry', 'type_wet', 'Temp_OXid', 'ppm', 'Pressure', 'Oxid_time', 'thickness']
X = df_test[oxidation].copy()
y = df_test['is_low_yield'].copy()

In [4]:
# =========================
# 1. 모델 불러오기
# =========================
oxi_model = lgb.Booster(model_file="../../model/oxi_lgbm.txt")

# =========================
# 2. 불량 예측 확률 계산
# =========================
bad_prob = oxi_model.predict(X)

result = X.copy()
result["bad_prob"] = bad_prob
result["y_true"] = y.values if hasattr(y, "values") else y

# =========================
# 3. 위험구간 기준 설정
# =========================
low_threshold = 0.3
high_ratio = 0.02

high_threshold = result["bad_prob"].quantile(1 - high_ratio)

print("저위험 기준 bad_prob <=", low_threshold)
print("고위험 기준 bad_prob >=", high_threshold)

# =========================
# 4. 위험구간 부여
# =========================
def assign_risk_group(prob):
    if prob <= low_threshold:
        return "저위험"
    elif prob >= high_threshold:
        return "고위험"
    else:
        return "중위험"

result["risk_group"] = result["bad_prob"].apply(assign_risk_group)

# =========================
# 5. 위험구간별 성능 요약
# =========================
risk_summary = (
    result
    .groupby("risk_group")
    .agg(
        data_count=("y_true", "count"),
        actual_defect_count=("y_true", "sum"),
        actual_defect_rate=("y_true", "mean"),
        mean_pred_prob=("bad_prob", "mean"),
        min_pred_prob=("bad_prob", "min"),
        max_pred_prob=("bad_prob", "max")
    )
    .reset_index()
)

risk_summary["data_ratio_percent"] = risk_summary["data_count"] / len(result) * 100
risk_summary["actual_defect_rate_percent"] = risk_summary["actual_defect_rate"] * 100
risk_summary["mean_pred_prob_percent"] = risk_summary["mean_pred_prob"] * 100

display(risk_summary)

# =========================
# 6. 결과 확인
# =========================
display(result[["bad_prob", "y_true", "risk_group"]].head())

저위험 기준 bad_prob <= 0.3
고위험 기준 bad_prob >= 0.999815582891559


,risk_group,data_count,actual_defect_count,actual_defect_rate,mean_pred_prob,min_pred_prob,max_pred_prob,data_ratio_percent,actual_defect_rate_percent,mean_pred_prob_percent
0,고위험,93,93,1.000000,0.999921,9.998156e-01,0.999999,2.014295,100.000000,99.992066
1,저위험,4041,26,0.006434,0.002681,1.787741e-08,0.296838,87.524366,0.643405,0.268057
2,중위험,483,465,0.962733,0.967054,3.087322e-01,0.999816,10.461339,96.273292,96.705375


,bad_prob,y_true,risk_group
0,2.867582e-04,0,저위험
1,4.115324e-06,0,저위험
2,9.996848e-01,1,중위험
3,1.686434e-05,0,저위험
4,8.372713e-07,0,저위험
